# Introduction to Pandas

<center><img src="https://geo-python-site.readthedocs.io/en/latest/_images/pandas_logo.png" height="300"/></center>

**Pandas** is one of the most powerful Python libraries for working with data. It gives you tools to load, clean, explore, and transform data — all in just a few lines of code. It was created by [Wes McKinney](https://en.wikipedia.org/wiki/Wes_McKinney), and its name comes from the term "**Pan**el **Da**ta", an econometrics term for datasets that track the same individuals over multiple time periods.

The core data structure in Pandas is the **DataFrame** — think of it like a spreadsheet or a database table, but with the full power of Python behind it. Almost everything we do as data scientists starts with getting our data into a DataFrame.

This notebook introduces you to DataFrames from the ground up: how to create them, how to load data into them, and how to explore and manipulate that data using Pandas' built-in tools.

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain what Pandas is and why it matters for data science

2. Create a Pandas **DataFrame** from a list of dictionaries or a list of lists

3. Load data from a CSV file into a DataFrame using `pd.read_csv()`

4. Explore a DataFrame using `.head()`, `.tail()`, `.info()`, `.describe()`, `.shape`, and `.columns`

5. Access specific rows and columns using bracket notation, `.loc[]`, and `.iloc[]`

6. Filter rows using **boolean masking** and the `.query()` method

7. Use **groupby**, **sorting**, and **column operations** to summarise and transform data

8. Handle missing values with `.fillna()` and `.dropna()`

---

## Pandas Import

Before we can use Pandas, we need to **import** it. This is a one-line step that gives our script access to everything in the library.

In [1]:
# Standard import — always use 'pd' as the alias
import pandas as pd

The line `import pandas as pd` loads the entire Pandas library and makes it accessible via the shorthand `pd`.

Using `pd` as the alias is **standard practice** across the Python data science community. Sticking to conventions like this makes your code easier for others to read and understand.

> **Tip:** Comments (lines starting with `#`) are notes for humans — Python ignores them. Use them freely to explain your thinking.

---

## Getting a DataFrame Object

There are two ways to get a DataFrame to work with:

1. **From data already in your Python program** — using the `pd.DataFrame()` constructor

2. **From an external file** — using `pd.read_csv()`, `pd.read_excel()`, and similar functions

Let's start with building DataFrames from scratch. See the [DataFrame docs](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html) for full reference.

### Method 1: From a List of Dictionaries

Pass a **list of dictionaries** to `pd.DataFrame()`. Each dictionary becomes **one row**. The dictionary keys become the **column names**.

In [2]:
# Create a DataFrame from a list of dictionaries
# Each dictionary becomes one row; the keys become column names
data_lst = [{'a': 1, 'b': 2, 'c': 3}, {'a': 4, 'b': 5, 'c': 6, 'd': 7}]
df_example = pd.DataFrame(data_lst)
df_example

,a,b,c,d
0,1,2,3,NaN
1,4,5,6,7.0


Notice that the first dictionary has no `'d'` key. Pandas fills the missing value with **`NaN`** (Not a Number) — the standard way Pandas represents missing data.

Now let's go further — what if *every* dictionary is missing some keys? Pandas still builds the full table, placing `NaN` wherever a value is absent.

**What do you expect the DataFrame below to look like?**

In [3]:
# Each dict has a different key — Pandas creates a column for every unique key it finds
data_lst = [{'a': 1}, {'b': 5}, {'c': 4}]
df_sparse = pd.DataFrame(data_lst)
df_sparse

,a,b,c
0,1.0,NaN,NaN
1,NaN,5.0,NaN
2,NaN,NaN,4.0


### Method 2: From a List of Lists

Pass a `data` argument (a list of lists) and a `columns` argument (a list of column names). Each inner list becomes **one row**.

The number of values in each list must match the number of column names you provide.

In [4]:
# Two rows of 3 values each, matched to 3 column names
data_vals = [[1, 2, 3], [4, 5, 6]]
data_cols = ['a', 'b', 'c']
df_example2 = pd.DataFrame(data=data_vals, columns=data_cols)
df_example2

,a,b,c
0,1,2,3
1,4,5,6


Unlike the list-of-dictionaries method, this approach is stricter: if any row has **more values** than the number of columns you declared, Pandas will raise a `ValueError`. The cell below triggers that error intentionally — run it to see what it looks like.

In [5]:
# This raises a ValueError: the second row has 3 values but only 2 columns are declared
data_vals = [[1, 2], [4, 5, 6]]
data_cols = ['a', 'b']
df_err = pd.DataFrame(data=data_vals, columns=data_cols)  # <-- will raise ValueError

ValueError: 2 columns passed, passed data had 3 columns

If a row has **fewer** values than the number of columns, Pandas is more forgiving — the missing trailing values are filled with `NaN`.

In [6]:
# First row has 2 values, but 3 columns are declared — the missing 'c' becomes NaN
data_vals = [[1, 2], [4, 5, 6]]
data_cols = ['a', 'b', 'c']
df_example3 = pd.DataFrame(data=data_vals, columns=data_cols)
df_example3

,a,b,c
0,1,2,NaN
1,4,5,6.0


### Method 3: Reading External Data

In practice, data lives in files — CSVs, Excel sheets, JSON, SQL databases. Pandas has a `read_*` function for each format. The most common is `pd.read_csv()`.

```python
# Basic usage — assumes column names are in the first row
df = pd.read_csv('my_data.csv')

# If there are no column headers in the file:
df = pd.read_csv('my_data.csv', header=None)

# Assign your own column names as you load:
df = pd.read_csv('my_data.csv', header=None, names=['col1', 'col2', 'col3'])

# If the file uses a separator other than a comma (e.g. semicolons):
df = pd.read_csv('my_data.csv', sep=';')
```

For the full list of options, see the [`pd.read_csv()` documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

---

## Looking at Our Data

Our dataset is `abalone.csv` — Let's load it and start exploring.

In [7]:
# Load the abalone dataset from the shared data folder
# The path goes one level up (../) because this notebook is inside the 1_Pandas/ folder
df = pd.read_csv('../data/abalone.csv')

A DataFrame has **attributes** and **methods** that help you understand what's inside it before you start working with it. The key ones are:

- **Attributes** (no parentheses): `.shape`, `.columns`

- **Methods** (with parentheses): `.head()`, `.tail()`, `.info()`, `.describe()`

Let's work through each one.

In [8]:
# Show the first 5 rows (default)
df.head()

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


With this data we are taking a trip under the sea! Abalones are large sea snails found along coastlines around the world.

<center><img src="https://upload.wikimedia.org/wikipedia/commons/b/bc/Abalone_at_California_Academy_of_Sciences.JPG" height="400"/></center>

The dataset comes from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Abalone). The original research goal was to **predict the age of an abalone** from physical measurements — a much faster (and kinder) alternative to the traditional method of cutting the shell and counting growth rings under a microscope.

**Column descriptions:**

| Column | Type | Unit | Description |
|:---|:---|:---|:---|
| `sex` | nominal | — | M (male), F (female), I (infant) |
| `length` | continuous | mm | Longest shell measurement |
| `diameter` | continuous | mm | Perpendicular to length |
| `height` | continuous | mm | With meat in shell |
| `weight_whole` | continuous | grams | Whole abalone |
| `weight_shucked` | continuous | grams | Weight of meat only |
| `viscera` | continuous | grams | Gut weight after bleeding |
| `shell` | continuous | grams | Shell weight after drying |
| `n_rings` | integer | — | Number of rings (+1.5 = age in years) |

In [9]:
# Pass a number to see more rows — useful when you want a bigger preview
df.head(10)

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7
5,I,0.425,0.300,0.095,0.3515,0.1410,0.0775,0.120,8
6,F,0.530,0.415,0.150,0.7775,0.2370,0.1415,0.330,20
7,F,0.545,0.425,0.125,0.7680,0.2940,0.1495,0.260,16
8,M,0.475,0.370,0.125,0.5095,0.2165,0.1125,0.165,9
9,F,0.550,0.440,0.150,0.8945,0.3145,0.1510,0.320,19


`.tail()` shows the **last** rows of the DataFrame — useful for checking that your data loaded completely.

In [10]:
# Show the last 5 rows
df.tail()

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10
4176,M,0.710,0.555,0.195,1.9485,0.9455,0.3765,0.4950,12


`.shape` returns a tuple `(rows, columns)`. `.columns` returns all column names as an **Index** object — similar to a list.

In [11]:
# (rows, columns) — no parentheses, it's an attribute not a method
df.shape

(4177, 9)

In [12]:
# Returns an Index of all column names
df.columns

Index(['sex', 'length', 'diameter', 'height', 'weight_whole', 'weight_shucked',
       'viscera', 'shell', 'n_rings'],
      dtype='str')

We have **4,177 rows** and **9 columns**.

`.info()` goes deeper — it shows the **data type** of each column and how many **non-null** values are present. This helps you quickly spot missing data and verify that columns loaded with the right types.

In [ ]:
# Summary of column types and non-null counts
df.info()

All 4,177 rows have values in every column — no missing data here.

`.describe()` gives you a **statistical summary** of all numeric columns: count, mean, standard deviation, min, max, and the three quartiles. It's a quick way to get a feel for the spread of your data.

In [13]:
# Statistical summary of all numeric columns
df.describe()

,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
count,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000
mean,0.523992,0.407881,0.139516,0.828742,0.359367,0.180594,0.238831,9.933684
std,0.120093,0.099240,0.041827,0.490389,0.221963,0.109614,0.139203,3.224169
min,0.075000,0.055000,0.000000,0.002000,0.001000,0.000500,0.001500,1.000000
25%,0.450000,0.350000,0.115000,0.441500,0.186000,0.093500,0.130000,8.000000
50%,0.545000,0.425000,0.140000,0.799500,0.336000,0.171000,0.234000,9.000000
75%,0.615000,0.480000,0.165000,1.153000,0.502000,0.253000,0.329000,11.000000
max,0.815000,0.650000,1.130000,2.825500,1.488000,0.760000,1.005000,29.000000


Notice that `sex` is missing from `.describe()` — it only summarises **numeric** columns. The `count` row tells you how many non-null values each column has. Since all counts equal 4,177, we have no missing values in this dataset.

---

## Accessing Data in a DataFrame

Now that we can load and inspect data, let's learn how to **select** specific parts of it.

There are several ways to grab data from a DataFrame, and the right approach depends on what you want:

- **A single column** → bracket notation or dot notation

- **Multiple columns** → pass a list inside brackets

- **A slice of rows** → slice notation

- **Specific rows AND columns** → `.loc[]` (label-based) or `.iloc[]` (integer-based)

### Selecting Columns

Use the **column name** in brackets to get an entire column. This returns a **Series** — a one-dimensional array with an index attached (more on this later).

In [14]:
# Bracket notation — always works
df['sex']

0       M
1       M
2       F
3       M
4       I
       ..
4172    F
4173    M
4174    M
4175    F
4176    M
Name: sex, Length: 4177, dtype: str

In [ ]:
# Dot notation — shorter, but has limitations (see note below)
df.sex

> **Note:** Dot notation (`df.sex`) fails if the column name contains **spaces** or clashes with an existing DataFrame method. Bracket notation (`df['sex']`) always works and is the safer choice.

If you ever load data with spaces in column names, here's a quick fix:

In [16]:
# Replace spaces with underscores in all column names
df2 = df.copy()
df2.columns = [col.replace(' ', '_') for col in df2.columns]
df2.columns
df2

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


To select **multiple columns** at once, pass a **list** of column names inside the brackets. This returns a DataFrame (not a Series).

In [17]:
# Select two columns at once — notice the double brackets [[ ]]
df[['sex', 'length']]

,sex,length
0,M,0.455
1,M,0.350
2,F,0.530
3,M,0.440
4,I,0.330
...,...,...
4172,F,0.565
4173,M,0.590
4174,M,0.600
4175,F,0.625


### Selecting Rows

You can **slice** rows using Python's standard slice notation — the same way you would slice a list.

In [18]:
# Rows 0, 1, 2 — up to but not including index 3
df[:3]

# A single integer raises a KeyError — Pandas looks for a column named 0, not row 0
# Use a slice or .loc/.iloc to get specific rows

# df[0]  # <-- uncomment and run to see: KeyError: 0

# You also can't select rows AND columns at the same time with plain brackets:

# df[:1, 'sex']  # <-- uncomment and run to see: TypeError

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.15,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.07,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.21,9


### Selecting Rows & Columns: `.loc[]` and `.iloc[]`

When you need to select **specific rows and specific columns at the same time**, use one of these two indexers:

| Indexer | Takes | Example |
|:---|:---|:---|
| `.loc[]` | **Labels** (column names, index labels) | `df.loc[0, 'sex']` |
| `.iloc[]` | **Integer positions** (0, 1, 2...) | `df.iloc[0, 0]` |

Think of it this way: `.loc` = **l**abels, `.iloc` = **i**ntegers.

In [19]:
# Single value: row with index label 0, column named 'sex'
df.loc[0, 'sex']

'M'

In [20]:
# Range of rows (inclusive on both ends with .loc), single column
df.loc[0:10, 'sex']

0     M
1     M
2     F
3     M
4     I
5     I
6     F
7     F
8     M
9     F
10    F
Name: sex, dtype: str

In [21]:
# Range of rows, multiple columns passed as a list
df.loc[10:15, ['sex', 'length']]

,sex,length
10,F,0.525
11,M,0.430
12,M,0.490
13,F,0.535
14,F,0.470
15,M,0.500


In [22]:
# .loc[] raises a KeyError when you pass integer column positions — it only accepts labels.
# Uncomment any of these to see the error:
# df.loc[0, 0]           # KeyError: 0 is not a column label
# df.loc[0:10, 0]        # KeyError
# df.loc[10:15, [0, 4]]  # KeyError

In [25]:
# .iloc[] works with integer positions — it does NOT accept column labels
print(df.iloc[0, 0])        # single value: row 0, column 0
print(df.iloc[0:5, 0])      # first 5 rows, first column
df.iloc[10:15, [0, 4]]      # rows 10-14, columns 0 and 4 by position

M
0    M
1    M
2    F
3    M
4    I
Name: sex, dtype: str


,sex,weight_whole
10,F,0.6065
11,M,0.4060
12,M,0.5415
13,F,0.6845
14,F,0.4755


In [26]:
# .iloc[] raises a ValueError when you pass a label — it only accepts integers.
# Uncomment to see the error:
# df.iloc[0, 'sex']  # ValueError: cannot label-index with a string using iloc

### Filtering with Boolean Masks and `.query()`

So far we've selected data by **position** or **label**. But often we want to select rows based on a **condition** — for example, "show me all abalones with a length under 0.4mm".

**Step 1:** A condition on a column returns a **boolean mask** — a Series of `True`/`False` values, one per row.

In [27]:
# This returns True/False for every row — not the data itself yet
df['length'] <= 0.4

0       False
1        True
2       False
3       False
4        True
        ...  
4172    False
4173    False
4174    False
4175    False
4176    False
Name: length, Length: 4177, dtype: bool

**Step 2:** Use the mask **inside brackets** to filter the DataFrame — only rows where the mask is `True` are returned.

In [28]:
# Only rows where length is 0.4 or less
df[df['length'] <= 0.4]

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
16,I,0.355,0.280,0.085,0.2905,0.0950,0.0395,0.1150,7
18,M,0.365,0.295,0.080,0.2555,0.0970,0.0430,0.1000,7
20,M,0.355,0.280,0.095,0.2455,0.0955,0.0620,0.0750,11
...,...,...,...,...,...,...,...,...,...
4151,I,0.350,0.250,0.075,0.1695,0.0835,0.0355,0.0410,6
4152,I,0.370,0.280,0.090,0.2180,0.0995,0.0545,0.0615,7
4162,M,0.385,0.255,0.100,0.3175,0.1370,0.0680,0.0920,8
4163,I,0.390,0.310,0.085,0.3440,0.1810,0.0695,0.0790,7


You can combine multiple conditions using `&` (AND) and `|` (OR). Each condition **must** be wrapped in its own parentheses.

In [29]:
# Female abalones with length 0.4 or less
df[(df['length'] <= 0.4) & (df['sex'] == 'F')]

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
71,F,0.400,0.320,0.110,0.3530,0.1405,0.0985,0.1000,8
132,F,0.325,0.260,0.090,0.1915,0.0850,0.0360,0.0620,7
136,F,0.305,0.230,0.080,0.1560,0.0675,0.0345,0.0480,7
226,F,0.390,0.290,0.125,0.3055,0.1210,0.0820,0.0900,7
331,F,0.400,0.325,0.120,0.3185,0.1340,0.0565,0.0950,8
514,F,0.275,0.195,0.070,0.0800,0.0310,0.0215,0.0250,5
519,F,0.345,0.250,0.090,0.2030,0.0780,0.0590,0.0550,6
521,F,0.360,0.270,0.090,0.1885,0.0845,0.0385,0.0550,5
539,F,0.375,0.290,0.115,0.2705,0.0930,0.0660,0.0885,10
544,F,0.380,0.290,0.105,0.2570,0.0990,0.0510,0.0850,10


In [30]:
# A more complex filter — multiple conditions combined with &
df[(df['length'] <= 0.4) & (df['length'] > 0.35) & (df['weight_whole'] > 0.25) & (df['weight_whole'] < 0.3)]

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
16,I,0.355,0.280,0.085,0.2905,0.0950,0.0395,0.1150,7
18,M,0.365,0.295,0.080,0.2555,0.0970,0.0430,0.1000,7
121,I,0.385,0.295,0.085,0.2535,0.1030,0.0575,0.0850,7
127,I,0.385,0.290,0.085,0.2505,0.1120,0.0610,0.0800,8
139,M,0.375,0.285,0.095,0.2530,0.0960,0.0575,0.0925,9
...,...,...,...,...,...,...,...,...,...
3923,M,0.395,0.280,0.080,0.2660,0.0995,0.0660,0.0900,12
3968,I,0.375,0.290,0.095,0.2875,0.1230,0.0605,0.0800,6
3969,I,0.380,0.300,0.090,0.2770,0.1655,0.0625,0.0820,6
3971,I,0.400,0.295,0.095,0.2520,0.1105,0.0575,0.0660,6


The `.query()` method lets you write the same filter as a **readable string**. It uses `and`/`or` instead of `&`/`|`, and drops the heavy bracket nesting. For complex filters, `.query()` is much easier to read.

In [31]:
# Same filter as above, written as a readable string
df.query('length <= 0.4 and length > 0.35 and weight_whole > 0.25 and weight_whole < 0.3')

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
16,I,0.355,0.280,0.085,0.2905,0.0950,0.0395,0.1150,7
18,M,0.365,0.295,0.080,0.2555,0.0970,0.0430,0.1000,7
121,I,0.385,0.295,0.085,0.2535,0.1030,0.0575,0.0850,7
127,I,0.385,0.290,0.085,0.2505,0.1120,0.0610,0.0800,8
139,M,0.375,0.285,0.095,0.2530,0.0960,0.0575,0.0925,9
...,...,...,...,...,...,...,...,...,...
3923,M,0.395,0.280,0.080,0.2660,0.0995,0.0660,0.0900,12
3968,I,0.375,0.290,0.095,0.2875,0.1230,0.0605,0.0800,6
3969,I,0.380,0.300,0.090,0.2770,0.1655,0.0625,0.0820,6
3971,I,0.400,0.295,0.095,0.2520,0.1105,0.0575,0.0660,6


For a complete list of DataFrame attributes and methods, see the [Pandas DataFrame documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html). For a deep-dive with real-world examples, the book [Python for Data Analysis](http://shop.oreilly.com/product/0636920023784.do) (by Wes McKinney, the creator of Pandas) is an excellent resource.

---

## Groupby, Sorting, and Column Operations

### Groupby

**Groupby** is one of the most powerful tools in Pandas. It follows a **split → apply → combine** pattern:

1. **Split** the DataFrame into groups based on a column#

2. **Apply** an aggregation function (mean, count, max, etc.) to each group

3. **Combine** the results back into a single DataFrame

This is equivalent to a `GROUP BY` clause in SQL or a pivot table in Excel.

In [32]:
# Check how many unique categories exist in the 'sex' column
df['sex'].unique()

<StringArray>
['M', 'F', 'I']
Length: 3, dtype: str

In [33]:
# Count the number of distinct values — useful to know before grouping
df['sex'].nunique()

3

In [34]:
# Calling groupby() alone returns a GroupBy object — it doesn't compute anything yet.
# You need to chain it with an aggregation method to get results.
df.groupby('sex')

Store the groupby object in a variable when you plan to run multiple aggregations on it — this is more efficient than calling `.groupby()` repeatedly.

In [35]:
# Store the groupby object for reuse
groupby_sex = df.groupby('sex')

# --- Mean of all numeric columns, grouped by sex ---
# Pandas 3.0 note: pass numeric_only=True to avoid warnings on non-numeric columns
groupby_sex.mean(numeric_only=True)

,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
sex,,,,,,,,
F,0.579093,0.454732,0.158011,1.046532,0.446188,0.230689,0.302010,11.129304
I,0.427746,0.326494,0.107996,0.431363,0.191035,0.092010,0.128182,7.890462
M,0.561391,0.439287,0.151381,0.991459,0.432946,0.215545,0.281969,10.705497


In [36]:
# Maximum value in each group
groupby_sex.max(numeric_only=True)

,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
sex,,,,,,,,
F,0.815,0.65,1.130,2.6570,1.4880,0.5900,1.005,29
I,0.725,0.55,0.220,2.0495,0.7735,0.4405,0.655,21
M,0.780,0.63,0.515,2.8255,1.3510,0.7600,0.897,27


In [37]:
# Count of non-null values per column in each group
groupby_sex.count()

,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
sex,,,,,,,,
F,1307,1307,1307,1307,1307,1307,1307,1307
I,1342,1342,1342,1342,1342,1342,1342,1342
M,1528,1528,1528,1528,1528,1528,1528,1528


The groupby column (`sex`) becomes the **index** of the result. To extract just one column from the grouped result — for example, the count per sex group — chain it with bracket notation.

In [41]:
# How many abalones of each sex are in the dataset?
df.groupby('sex').count()['length']

sex
F    1307
I    1342
M    1528
Name: length, dtype: int64

You can group by **multiple columns** by passing a list. Pandas groups by the first column, then by the second within each of those groups — creating a **MultiIndex** result.

In [ ]:
# Group by sex, then by ring count — how many abalones in each sex/ring combination?
df.groupby(['sex', 'n_rings']).count()['weight_whole']

sex  n_rings
F    5            4
     6           16
     7           44
     8          122
     9          238
               ... 
M    22           3
     23           3
     24           1
     26           1
     27           1
Name: length, Length: 68, dtype: int64

See all available aggregation options in the [Groupby documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html).

### Sorting

`.sort_values()` sorts the DataFrame by one or more columns. By default it sorts **ascending** (smallest first).

In [44]:
# Sort by length ascending (default), show the 15 smallest abalones
df.sort_values('length').head(15)

# To reverse the order: df.sort_values('length', ascending=False)

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
236,I,0.075,0.055,0.010,0.0020,0.0010,0.0005,0.0015,1
238,I,0.110,0.090,0.030,0.0080,0.0025,0.0020,0.0030,3
237,I,0.130,0.100,0.030,0.0130,0.0045,0.0030,0.0040,3
2114,I,0.130,0.095,0.035,0.0105,0.0050,0.0065,0.0035,4
1986,I,0.135,0.130,0.040,0.0290,0.0125,0.0065,0.0080,4
1429,I,0.140,0.105,0.035,0.0140,0.0055,0.0025,0.0040,3
3899,I,0.140,0.105,0.035,0.0145,0.0050,0.0035,0.0050,4
719,I,0.150,0.100,0.025,0.0150,0.0045,0.0040,0.0050,2
526,M,0.155,0.110,0.040,0.0155,0.0065,0.0030,0.0050,3
2381,M,0.155,0.115,0.025,0.0240,0.0090,0.0050,0.0075,5


To sort by **multiple columns**, pass a list. Pandas sorts by the first column first, then uses the second to break ties.

In [45]:
# Sort by length descending, then diameter descending (to break ties)
df.sort_values(['length', 'diameter'], ascending=False).head(15)

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,n_rings
1428,F,0.815,0.650,0.250,2.2550,0.8905,0.4200,0.7975,14
2334,F,0.800,0.630,0.195,2.5260,0.9330,0.5900,0.6200,23
1209,F,0.780,0.630,0.215,2.6570,1.4880,0.4985,0.5860,11
3715,M,0.780,0.600,0.210,2.5480,1.1945,0.5745,0.6745,11
1763,M,0.775,0.630,0.250,2.7795,1.3485,0.7600,0.5780,12
2090,M,0.775,0.570,0.220,2.0320,0.7350,0.4755,0.6585,17
1762,M,0.770,0.620,0.195,2.5155,1.1155,0.6415,0.6420,12
4148,M,0.770,0.605,0.175,2.0505,0.8005,0.5260,0.3550,11
2625,M,0.770,0.600,0.215,2.1945,1.0515,0.4820,0.5840,10
1052,M,0.765,0.600,0.220,2.3020,1.0070,0.5090,0.6205,12


### Creating and Dropping Columns

You can **create** a new column using:
1. **Direct assignment**: `df['new_col'] = expression`
2. **`df.eval()`**: write the expression as a string — useful for readable, concise column creation

You can **drop** a column with `df.drop()`. Remember to specify `axis=1` (columns) vs `axis=0` (rows).

![Axis diagram](https://i.stack.imgur.com/DL0iQ.jpg)

In [46]:
# Rename the 'n_rings' column to 'nr_rings' for clarity
df.rename(columns={'n_rings': 'nr_rings'}, inplace=True)
df.head()

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,nr_rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [48]:
# Create a new 'age' column — per the dataset: age (years) = nr_rings + 1.5
# Pandas 3.0 note: eval() with inplace=True is deprecated — assign the result directly
df = df.eval('age = nr_rings + 1.5')
df.columns
df.head()

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,nr_rings,age
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,16.5
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,8.5
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,10.5
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,11.5
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,8.5


In [49]:
# Create another derived column: weight relative to diameter
# Pandas 3.0 note: eval() with inplace=True is deprecated — assign the result directly
df = df.eval('weight_per_diam = weight_whole / diameter')
df.columns

Index(['sex', 'length', 'diameter', 'height', 'weight_whole', 'weight_shucked',
       'viscera', 'shell', 'nr_rings', 'age', 'weight_per_diam'],
      dtype='str')

In [50]:
# Verify the new columns were created correctly
df.head()

,sex,length,diameter,height,weight_whole,weight_shucked,viscera,shell,nr_rings,age,weight_per_diam
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,16.5,1.408219
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,8.5,0.850943
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,10.5,1.611905
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,11.5,1.413699
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,8.5,0.803922


Now that we have `age`, the `nr_rings` column is redundant — both encode the same information. Let's drop it.

By default, `df.drop()` returns a **new** DataFrame without the column, leaving the original unchanged. Pass `inplace=True` to modify `df` directly.

In [57]:
# Without inplace=True, the result is returned but df is NOT modified
df.drop('nr_rings', axis=1).columns


KeyError: "['nr_rings'] not found in axis"

In [52]:
# nr_rings is still in df — the drop was not applied
df.columns

Index(['sex', 'length', 'diameter', 'height', 'weight_whole', 'weight_shucked',
       'viscera', 'shell', 'nr_rings', 'age', 'weight_per_diam'],
      dtype='str')

In [54]:
# Now drop it for real — inplace=True modifies df directly
df.drop('nr_rings', axis=1, inplace=True)

KeyError: "['nr_rings'] not found in axis"

In [58]:
# Confirm nr_rings is gone
df.columns

Index(['sex', 'length', 'diameter', 'height', 'weight_whole', 'weight_shucked',
       'viscera', 'shell', 'age', 'weight_per_diam'],
      dtype='str')

### Dealing with Null Values

Real-world datasets almost always have **missing values** (represented as `NaN` in Pandas). You have two main options:

- **`.fillna(value)`** — replace `NaN` with a specific value (e.g. `0`, `-1`, or the column mean)

- **`.dropna()`** — remove any row that contains a `NaN`

The right choice depends on your data and your goal. Read more in the [Missing Data guide](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html).

In [59]:
# Our abalone dataset has no missing values, so these lines have no visible effect here.
# They demonstrate the pattern you'll use on datasets that do have NaNs.

# --- Replace NaN with a placeholder value ---
df.fillna(-1, inplace=True)

# --- Drop any row that contains a NaN ---
df.dropna(inplace=True)

---

## A Quick Aside: Pandas Series

You may have noticed that selecting a single column returns something slightly different from a DataFrame — it has a column of values with an index attached. This is called a **`Series`**.

A **Series** is a one-dimensional labelled array. Think of it as a single column from a DataFrame. Most DataFrame methods also work on Series.

Here are some examples from cells we've already run:

In [60]:
# A boolean condition on a column returns a Series of True/False values
df['length'] <= 0.3

0       False
1       False
2       False
3       False
4       False
        ...  
4172    False
4173    False
4174    False
4175    False
4176    False
Name: length, Length: 4177, dtype: bool

In [61]:
# Confirm it's a Series object
type(df['length'] <= 0.3)

pandas.Series

In [62]:
# Selecting one column from a groupby result also returns a Series
df.groupby('sex').count()['age']

sex
F    1307
I    1342
M    1528
Name: age, dtype: int64

---

## Key Takeaways

- **Pandas** is the standard library for working with tabular data in Python

- The **DataFrame** is your main data structure — think of it as a programmable spreadsheet

- Always **explore before you transform**: use `.head()`, `.info()`, `.describe()`, `.shape`

- Use **bracket notation** or `.loc[]` / `.iloc[]` to access rows and columns

- **Boolean masking** and `.query()` let you filter rows by condition

- **Groupby** aggregates data across categories (split → apply → combine)

- Handle missing data with `.fillna()` or `.dropna()` — never ignore `NaN`s

---

## Check Your Understanding

Answer these questions mentally or in a scratch cell before moving on:

1. What is the difference between `.loc[]` and `.iloc[]`? Give one example of each.

2. You run `df['age'].mean()` and get `NaN`. What is the most likely cause?

3. What does `.describe()` NOT show, and why?

4. You want to find all rows where `weight_whole > 1.0` AND `sex == 'M'`. Write the boolean mask filter.

5. What is the difference between `df.drop('col', axis=1)` and `df.drop('col', axis=1, inplace=True)`?

Answers:
1. .loc is for labeled column values and .iloc for numeric values.
2. They are not grouped.
3. describe() show summary of aggregated functions and not of string or categorical variables.
4. df.query('weight_whole > 1.0 and sex == "M"')
5. df.drop('col', axis=1) does not really drop the column in the DF, only in the results, and `df.drop('col', axis=1, inplace=True) actually does it. However, when I run the example inplace=True was not necessary for dropping the column. Perhaps is due to an update in the panda version.



### **Ready to practice?** Open [Notebook_2](02_Pandas_Practice_Functionalities.ipynb) to apply what you've learned.

---

## Further Study

There are many more methods and features in Pandas. Here are some resources to go deeper:

- [Helpful Python Code Snippets for Data Exploration in Pandas](https://medium.com/@msalmon00/helpful-python-code-snippets-for-data-exploration-in-pandas-b7c5aed5ecb9)

- [Manipulating tabular data with Pandas](https://neuroimaging-data-science.org/content/004-scipy/002-pandas.html)

- [Book — Python for Data Analysis (Wes McKinney)](http://shop.oreilly.com/product/0636920023784.do)

- [Pandas official documentation](https://pandas.pydata.org/pandas-docs/stable/)